# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [81]:
import joblib
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, ParameterGrid, cross_val_score
from tqdm.notebook import tqdm
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [90]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
    
    def transform(self,X):
        
        X_transformed = X.copy()
        X_transformed['timestamp'] = pd.to_datetime(X_transformed['timestamp'])
        X_transformed['hour'] = X_transformed['timestamp'].dt.hour 
        X_transformed['dayofweek'] = X_transformed['timestamp'].dt.weekday
        X_transformed = X_transformed.drop(['timestamp'], axis=1)
        return X_transformed
    

In [55]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    
    def __init__(self, target_column):
        self.target_column = target_column
        self.encoder = None
        self.categorical_columns = None
        self.encoded_feature_names = None
    
    def fit(self, X, y=None):
        
        X_copy = X.copy()
        
        self.categorical_columns = X_copy.select_dtypes(
            include=['str', 'category']
        ).columns.tolist()
        
        if self.target_column in self.categorical_columns:
            self.categorical_columns.remove(self.target_column)
        
        if self.categorical_columns:
            self.encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            self.encoder.fit(X_copy[self.categorical_columns])
            self.encoded_feature_names = self.encoder.get_feature_names_out(
                self.categorical_columns
            )
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        target = X_copy[self.target_column]
        X_features = X_copy.drop(columns=[self.target_column])
        
        if self.categorical_columns and self.encoder is not None:
            
            encoded_array = self.encoder.transform(X_features[self.categorical_columns])
            
            encoded_df = pd.DataFrame(encoded_array, columns=self.encoded_feature_names, index=X_features.index)
            
            X_features = X_features.drop(columns=self.categorical_columns)
            X_features = pd.concat([X_features, encoded_df], axis=1)
            
        return X_features, target
    
            

        
    

In [56]:
df = pd.read_csv('../data/checker_submits.csv')
print(df.head(3))

ppl = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder(target_column='dayofweek'))]) 

X, y = ppl.fit_transform(df)
print('\n',X.head(3))
print('\n',y.head(3))

      uid   labname  numTrials                   timestamp
0  user_4  project1          1  2020-04-17 05:19:02.744528
1  user_4  project1          2  2020-04-17 05:22:45.549397
2  user_4  project1          3  2020-04-17 05:34:24.422370

    numTrials  hour  uid_user_0  uid_user_1  uid_user_10  uid_user_11  \
0          1     5         0.0         0.0          0.0          0.0   
1          2     5         0.0         0.0          0.0          0.0   
2          3     5         0.0         0.0          0.0          0.0   

   uid_user_12  uid_user_13  uid_user_14  uid_user_15  ...  labname_lab02  \
0          0.0          0.0          0.0          0.0  ...            0.0   
1          0.0          0.0          0.0          0.0  ...            0.0   
2          0.0          0.0          0.0          0.0  ...            0.0   

   labname_lab03  labname_lab03s  labname_lab05s  labname_laba04  \
0            0.0             0.0             0.0             0.0   
1            0.0            

In [57]:
class TrainValidationTest(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y):
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, test_size=0.2, random_state=21, stratify=y
        )
        
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_temp, y_temp, test_size=0.25, random_state=21, stratify=y_temp
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test
    
    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X,  y)

In [58]:
tvt = TrainValidationTest()

X_train, X_valid, X_test, y_train, y_valid, y_test = tvt.fit_transform(X,y)

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [62]:
class ModelSelection:
    
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self._results: list[dict] = []          

    
    def choose(self,X_train,y_train,X_valid,y_valid):
        self._results = []
        best_overall_score = -1.0
        best_overall_idx = None
        for idx, grid in enumerate(self.grids):
            name       = self.grid_dict[idx]
            estimator  = grid.estimator
            param_grid = grid.param_grid
            scoring    = grid.scoring
            cv         = grid.cv
            n_jobs     = grid.n_jobs
            
            
            print(f"Estimator: {name}")

            param_list   = list(ParameterGrid(param_grid))
            best_cv      = -1.0
            best_params  = None

            for params in tqdm(param_list):
                model = clone(estimator).set_params(**params)
                cv_scores = cross_val_score(
                    model, X_train, y_train,
                    scoring=scoring, cv=cv, n_jobs=n_jobs,
                )
                mean_cv = cv_scores.mean()

                if mean_cv > best_cv:
                    best_cv     = mean_cv
                    best_params = params

            best_model = clone(estimator).set_params(**best_params)
            best_model.fit(X_train, y_train)
            valid_score = best_model.score(X_valid, y_valid)

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_cv:.3f}")
            print(
                f"Validation set accuracy score for best params: "
                f"{valid_score:.3f} \n"
            )

            self._results.append(
                {
                    "model": name,
                    "params": best_params,
                    "valid_score": valid_score,
                }
            )

            if valid_score > best_overall_score:
                best_overall_score = valid_score
                best_overall_idx   = idx

        winner = self.grid_dict[best_overall_idx]
        print(f"Classifier with best validation set accuracy: {winner}")
        return winner

    def best_results(self) -> pd.DataFrame:
        
        if not self._results:
            raise RuntimeError("Call .choose() before .best_results()")
        return pd.DataFrame(self._results)            

        
    
    

In [63]:
jobs = -1

svm_params = [
    {
        "kernel": ("linear", "rbf", "sigmoid"),
        "C": [0.01, 0.1, 1, 1.5, 5, 10],
        "gamma": ["scale", "auto"],
        "class_weight": ("balanced", None),
        "random_state": [21],
        "probability": [True],
    }
]
gs_svm = GridSearchCV(
    estimator=SVC(),
    param_grid=svm_params,
    scoring="accuracy",
    cv=2,
    n_jobs=jobs,
)

tree_params = [
    {
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 3, 5, 10, 21],
        "class_weight": ["balanced", None],
        "random_state": [21],
    }
]
gs_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=tree_params,
    scoring="accuracy",
    cv=2,
    n_jobs=jobs,
)

rf_params = [
    {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 3, 5, 10, 22],
        "criterion": ["gini", "entropy"],
        "class_weight": ["balanced", None],
        "random_state": [21],
    }
]
gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=rf_params,
    scoring="accuracy",
    cv=2,
    n_jobs=jobs,
)

grids     = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: "SVM", 1: "Decision Tree", 2: "Random Forest"}

selector  = ModelSelection(grids, grid_dict)
best_name = selector.choose(X_train, y_train, X_valid, y_valid)
selector.best_results()


Estimator: SVM


  0%|          | 0/72 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.752
Validation set accuracy score for best params: 0.855 

Estimator: Decision Tree


  0%|          | 0/20 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'random_state': 21}
Best training accuracy: 0.802
Validation set accuracy score for best params: 0.864 

Estimator: Random Forest


  0%|          | 0/60 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'n_estimators': 200, 'random_state': 21}
Best training accuracy: 0.858
Validation set accuracy score for best params: 0.884 

Classifier with best validation set accuracy: Random Forest


,model,params,valid_score
0,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.854599
1,Decision Tree,"{'class_weight': None, 'criterion': 'gini', 'm...",0.863501
2,Random Forest,"{'class_weight': None, 'criterion': 'gini', 'm...",0.884273


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [77]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator
    
    def final_score(self, X_train, y_train, X_test, y_test):
        estimator = self.estimator
        estimator.fit(X_train, y_train)
        acc = estimator.score(X_test, y_test)
        print(f"Accuracy of the final model is {acc}")
        return acc
    
    def save_model(self, path):
        estimator = self.estimator
        joblib.dump(estimator, path)
        
        

In [82]:
svc = SVC().set_params(**selector.best_results().iloc[0].params)
finalize = Finalize(svc)
finalize.final_score(X_train, y_train, X_test, y_test)
finalize.save_model("../models/finalize_test.joblib")

Accuracy of the final model is 0.8609467455621301


## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [86]:
df = pd.read_csv('../data/checker_submits.csv')
df.head(3)

,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370


In [91]:
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), 
                          ('onehot_encoder', MyOneHotEncoder(target_column='dayofweek'))]) 

In [115]:
X,y = preprocessing.fit_transform(df)
display(X.head(1))
display(y.head(1))

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


0    4
Name: dayofweek, dtype: int32

In [96]:
tvt = TrainValidationTest()
X_train,X_valid,X_test,y_train,y_valid,y_test = tvt.fit_transform(X,y)

In [107]:
jobs = -1

svm_params = [
    {'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'gamma': ['scale', 'auto'],
    'class_weight': [None, 'balanced'],
    }
]
gs_svm = GridSearchCV(
    estimator=SVC(),
    param_grid=svm_params,
    scoring="accuracy",
    cv=2,
    n_jobs=jobs,
)

tree_params = [
    {'max_depth': [i for i in range(1, 49+1)],
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy'],}
]
gs_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=tree_params,
    scoring="accuracy",
    cv=2,
    n_jobs=jobs,
)

rf_params = [
    {'max_depth': [i for i in range(1, 49+1)],
                   'class_weight': [None, 'balanced'],
                   'criterion': ['gini', 'entropy'],
                   'n_estimators': [5,10,50,100]}
]
gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=rf_params,
    scoring="accuracy",
    cv=2,
    n_jobs=jobs,
)

grids     = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: "SVM", 1: "Decision Tree", 2: "Random Forest"}

selector  = ModelSelection(grids, grid_dict)
best_name = selector.choose(X_train, y_train, X_valid, y_valid)
selector.best_results()


Estimator: SVM


  0%|          | 0/72 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Best training accuracy: 0.752
Validation set accuracy score for best params: 0.855 

Estimator: Decision Tree


  0%|          | 0/196 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 38}
Best training accuracy: 0.819
Validation set accuracy score for best params: 0.869 

Estimator: Random Forest


  0%|          | 0/784 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 26, 'n_estimators': 100}
Best training accuracy: 0.862
Validation set accuracy score for best params: 0.875 

Classifier with best validation set accuracy: Random Forest


,model,params,valid_score
0,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.854599
1,Decision Tree,"{'class_weight': None, 'criterion': 'gini', 'm...",0.869436
2,Random Forest,"{'class_weight': 'balanced', 'criterion': 'gin...",0.875371


In [108]:
best_model = RandomForestClassifier().set_params(**selector.best_results().iloc[2].params)
final = Finalize(best_model)
acc = final.final_score(X_train, y_train, X_test, y_test)
final.save_model(f"../models/rndForest_{acc:.3f}.sav")

Accuracy of the final model is 0.9142011834319527
